In [10]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader = TextLoader("speech.txt")
documents =loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000,chunk_overlap=30)
docs = text_splitter.split_documents(documents)

In [11]:
docs


[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…'),
 Document(metadata={'source': 'speech.txt'}, page_content='…\n\nIt will be all the easier for us to conduct our

In [13]:
embeddings = OllamaEmbeddings(model ="nomic-embed-text")
db = FAISS.from_documents(docs,embeddings)
db

In [16]:
query = "how does the speaker describe the desired outcome of the war?"
docs = db.similarity_search(query)
docs[0].page_content

'It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'

## as a retriever

In [17]:
retriever = db.as_retriever()
retriever.invoke(query)
docs[0].page_content

'It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'

## Similarity search with score

In [19]:
docs_and_score = db.similarity_search_with_score(query)
docs_and_score


[(Document(id='d72446da-8865-44d6-a7a9-23147cc98a1e', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
  np.float32(346.33426)),
 (Document(id='ebf6be2e-ff43-4681-b2a2-f10a74f68994', metadata={'source': 'spe

In [22]:
embedding_vector = embeddings.embed_query(query)
embedding_vector

[-0.3021728992462158,
 0.27879059314727783,
 -3.613856792449951,
 -0.9976816773414612,
 2.01922607421875,
 1.4693918228149414,
 -1.0232887268066406,
 0.23463353514671326,
 0.48269081115722656,
 -0.18005630373954773,
 -0.06285097450017929,
 0.5969094634056091,
 0.8666642904281616,
 1.6234604120254517,
 2.087756872177124,
 -0.780638575553894,
 0.25883030891418457,
 -1.2077631950378418,
 -0.6833815574645996,
 0.726655900478363,
 -0.9000466465950012,
 -0.39296936988830566,
 0.09221556782722473,
 0.47901052236557007,
 1.3452236652374268,
 0.41081467270851135,
 -0.5247780680656433,
 0.6715157628059387,
 -1.0716969966888428,
 -0.52569580078125,
 0.9153724312782288,
 0.06207527592778206,
 -0.05118250846862793,
 0.28220033645629883,
 -1.5354228019714355,
 -1.9183907508850098,
 0.07962840795516968,
 1.3347282409667969,
 0.35618796944618225,
 -1.4278382062911987,
 0.06117691472172737,
 -0.4963439702987671,
 0.12226945161819458,
 -1.5524982213974,
 1.366471290588379,
 -0.6894099712371826,
 -0.1292

In [23]:
docs_score = db.similarity_search_by_vector(embedding_vector)
docs_score

[Document(id='d72446da-8865-44d6-a7a9-23147cc98a1e', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
 Document(id='ebf6be2e-ff43-4681-b2a2-f10a74f68994', metadata={'source': 'speech.txt'}, page_content='…\n

In [26]:
db.save_local("FAISS_index")
new_db = FAISS.load_local(
    "FAISS_index",
    embeddings,
    allow_dangerous_deserialization=True,
)

In [28]:
docs = new_db.similarity_search(query)
docs

[Document(id='d72446da-8865-44d6-a7a9-23147cc98a1e', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
 Document(id='ebf6be2e-ff43-4681-b2a2-f10a74f68994', metadata={'source': 'speech.txt'}, page_content='…\n